# Oil Spill Detection — Sentinel-1 Training Notebook
### LinkNet + ResNet34, Refined Deep-SAR (SOS) dataset

**Read this before running anything.**

This notebook trains a binary oil-spill segmentation model on the **Sentinel-1-only** split of the Refined Deep-SAR SOS dataset (3354 train / 839 val images), using LinkNet with a ResNet34 encoder pretrained on ImageNet.

**Before you start:**
1. Upload your `SOS` dataset folder (with `images/train`, `images/val`, `masks/train`, `masks/val`) somewhere this notebook can read it — either:
   - **Google Colab**: zip the `SOS` folder locally, upload it to Google Drive, then mount Drive below and point `DATA_ROOT` at it, OR upload the zip directly via the Colab file browser and unzip it in Cell 3.
   - **Local Jupyter**: just point `DATA_ROOT` at your existing path, e.g. `C:/Users/datha/Desktop/sar/dataset/SOS`.
2. If on Colab: **Runtime → Change runtime type → GPU** (T4 is fine, has more VRAM than your local 6GB card, so you *could* try a larger batch size here — but start at 4 anyway to keep results comparable to your local runs).
3. This notebook downloads the trained weights (`best_model.pth`) at the very end. Don't close the tab before that cell runs, or you'll lose the trained weights when the Colab runtime recycles.

**What this notebook does, in order:** environment check → install deps → load dataset → sanity-visualize a few samples → build model → train with checkpointing → evaluate (IoU/Dice/precision/recall/F1) → plot a few predictions → **download the weights file**.


## 1. Environment check
Confirms GPU availability and library versions before doing anything expensive.

In [ ]:
import sys, torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"Total VRAM: {props.total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: no GPU detected. Training will be very slow on CPU.")


## 2. Install dependencies
Only needed if these aren't already installed in your environment (Colab needs this every fresh session; a persistent local env usually doesn't).

In [ ]:
!pip install -q segmentation-models-pytorch albumentations


## 3. Dataset location
Set `DATA_ROOT` to wherever your `SOS` folder lives.

**Colab + Google Drive:**
```python
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = "/content/drive/MyDrive/sar/dataset/SOS"
```

**Colab + direct zip upload:** upload `SOS.zip` via the file browser on the left, then:
```python
!unzip -q SOS.zip -d /content/dataset
DATA_ROOT = "/content/dataset/SOS"
```

**Local Jupyter (Windows):**
```python
DATA_ROOT = r"C:\Users\datha\Desktop\sar\dataset\SOS"
```

Edit the line below to match your situation, then run it.

In [ ]:
DATA_ROOT = "/content/dataset/SOS"  # <-- EDIT THIS LINE

import os
assert os.path.isdir(DATA_ROOT), f"DATA_ROOT does not exist: {DATA_ROOT}. Fix the path above."
for sub in ["images/train", "images/val", "masks/train", "masks/val"]:
    p = os.path.join(DATA_ROOT, sub)
    assert os.path.isdir(p), f"Missing expected subfolder: {p}"
print("DATA_ROOT looks correct:", DATA_ROOT)


## 4. Dataset class (Sentinel-1 only)
Filters to `sentinel_*.png` only, matches image/mask pairs by relative path within each split (not a global stem set — that caused a duplicate-stem bug during dataset inspection), binarizes masks to {0,1}, and applies synchronized augmentation to image+mask via albumentations.

In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

class SARSentinelDataset(Dataset):
    def __init__(self, root, split, transform=None):
        self.root = Path(root)
        self.img_dir = self.root / "images" / split
        self.mask_dir = self.root / "masks" / split

        img_files = sorted(self.img_dir.glob("sentinel_*.png"))
        pairs, missing = [], []
        for img_path in img_files:
            mask_path = self.mask_dir / img_path.name
            if mask_path.exists():
                pairs.append((img_path, mask_path))
            else:
                missing.append(img_path.name)

        if missing:
            print(f"[WARNING] {len(missing)} sentinel images in '{split}' have no matching mask.")

        self.pairs = pairs
        self.transform = transform
        if len(self.pairs) == 0:
            raise RuntimeError(f"No sentinel_*.png pairs found in {self.img_dir}")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        mask = np.array(Image.open(mask_path).convert("L"))
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug["image"], aug["mask"]
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
            mask = torch.from_numpy(mask).float()

        if mask.dim() == 2:
            mask = mask.unsqueeze(0)
        return image, mask


def get_transforms(train):
    if train:
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def get_dataloaders(root, batch_size=4, num_workers=2):
    train_ds = SARSentinelDataset(root, "train", transform=get_transforms(True))
    val_ds = SARSentinelDataset(root, "val", transform=get_transforms(False))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader

train_loader, val_loader = get_dataloaders(DATA_ROOT, batch_size=4, num_workers=2)
print(f"Train batches: {len(train_loader)} (images: {len(train_loader.dataset)})")
print(f"Val batches:   {len(val_loader)} (images: {len(val_loader.dataset)})")


## 5. Sanity check: shapes, value ranges, and oil-pixel fraction
**Do not skip this.** If the oil-pixel fraction is very low (a few %), we'll weight the loss accordingly in Step 7. If shapes/values look wrong here, everything downstream is wasted compute.

In [ ]:
images, masks = next(iter(train_loader))
print("Image batch shape:", images.shape, images.dtype)
print("Mask batch shape:", masks.shape, masks.dtype)
print("Image min/max:", images.min().item(), images.max().item())
print("Mask unique values:", torch.unique(masks).tolist())

# Average oil-pixel fraction across a few batches
fractions = []
for i, (_, m) in enumerate(train_loader):
    fractions.append(m.mean().item())
    if i >= 9:
        break
oil_fraction = sum(fractions) / len(fractions)
print(f"Average oil-pixel fraction (10 batches): {oil_fraction:.4f}")


## 6. Visualize a few image/mask pairs
Confirms the mask actually lines up with visible features in the SAR image (dark, smooth patches = likely oil). If these look randomly shifted/rotated relative to each other, augmentation syncing is broken — stop and fix Step 4 before training.

In [ ]:
import matplotlib.pyplot as plt

raw_ds = SARSentinelDataset(DATA_ROOT, "train", transform=None)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    img, mask = raw_ds[i]
    axes[0, i].imshow(img.permute(1, 2, 0).numpy())
    axes[0, i].set_title(f"image {i}")
    axes[0, i].axis("off")
    axes[1, i].imshow(mask.squeeze(0).numpy(), cmap="gray")
    axes[1, i].set_title(f"mask {i}")
    axes[1, i].axis("off")
plt.tight_layout()
plt.savefig("sample_check.png", dpi=120)
plt.show()


## 7. Model, loss, and metrics

- **Model**: LinkNet + ResNet34 encoder (ImageNet pretrained). Lightweight enough for 6GB local VRAM; here on Colab GPU it'll have headroom to spare.
- **Loss**: 0.5 × Dice + 0.5 × weighted BCE. If oil pixels are a small minority (see Step 5's fraction), `pos_weight` in BCE compensates so the model doesn't just predict all-background.
- **Metrics**: IoU, Dice, precision, recall, F1 — **not** raw pixel accuracy, since a model that predicts "all background" would score deceptively high on accuracy alone.

In [ ]:
import segmentation_models_pytorch as smp
import torch.nn as nn

def build_model():
    return smp.Linknet(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
    )

model = build_model()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# quick forward-pass check
with torch.no_grad():
    dummy = torch.randn(2, 3, 256, 256).to(device)
    out = model(dummy)
    print("Model output shape:", out.shape)  # expect [2, 1, 256, 256]


In [ ]:
from segmentation_models_pytorch.losses import DiceLoss

dice_loss_fn = DiceLoss(mode="binary")

# pos_weight compensates for class imbalance (see oil_fraction from Step 5)
pos_weight_value = min((1 - oil_fraction) / max(oil_fraction, 1e-6), 10.0)
print(f"Using BCE pos_weight = {pos_weight_value:.2f} (capped at 10.0)")
bce_loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value).to(device))

def combined_loss(logits, targets):
    return 0.5 * dice_loss_fn(logits, targets) + 0.5 * bce_loss_fn(logits, targets)


def compute_metrics(logits, targets, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    tp = (preds * targets).sum()
    fp = (preds * (1 - targets)).sum()
    fn = ((1 - preds) * targets).sum()
    tn = ((1 - preds) * (1 - targets)).sum()

    eps = 1e-7
    iou = tp / (tp + fp + fn + eps)
    dice = (2 * tp) / (2 * tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = (2 * precision * recall) / (precision + recall + eps)

    return {
        "iou": iou.item(),
        "dice": dice.item(),
        "precision": precision.item(),
        "recall": recall.item(),
        "f1": f1.item(),
    }


## 8. Training loop
Uses AMP mixed precision to fit comfortably in 6GB-class VRAM. Saves the checkpoint with the **best validation IoU** to `best_model.pth` — that's the file you'll download at the end.

`EPOCHS` is set conservatively given the 2-day deadline context — increase it if you have GPU time to spare and validation IoU is still improving when it finishes.

In [ ]:
import csv
import time

EPOCHS = 30           # increase if val IoU is still improving and you have time
LR = 1e-4
PATIENCE = 8          # early stopping patience on val IoU
CHECKPOINT_PATH = "best_model.pth"
LOG_PATH = "training_log.csv"

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

best_iou = 0.0
epochs_without_improvement = 0

with open(LOG_PATH, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "val_loss", "val_iou", "val_dice", "val_precision", "val_recall", "val_f1", "time_sec"])

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # ---- train ----
    model.train()
    train_losses = []
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(images)
            loss = combined_loss(logits, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_losses.append(loss.item())

    # ---- validate ----
    model.eval()
    val_losses = []
    metric_totals = {"iou": 0.0, "dice": 0.0, "precision": 0.0, "recall": 0.0, "f1": 0.0}
    n_batches = 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(images)
                loss = combined_loss(logits, masks)
            val_losses.append(loss.item())
            m = compute_metrics(logits, masks)
            for k in metric_totals:
                metric_totals[k] += m[k]
            n_batches += 1

    val_metrics = {k: v / n_batches for k, v in metric_totals.items()}
    train_loss_avg = sum(train_losses) / len(train_losses)
    val_loss_avg = sum(val_losses) / len(val_losses)
    elapsed = time.time() - t0

    print(f"Epoch {epoch:02d}/{EPOCHS} | train_loss {train_loss_avg:.4f} | "
          f"val_loss {val_loss_avg:.4f} | val_IoU {val_metrics['iou']:.4f} | "
          f"val_Dice {val_metrics['dice']:.4f} | {elapsed:.1f}s")

    with open(LOG_PATH, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([epoch, train_loss_avg, val_loss_avg, val_metrics["iou"], val_metrics["dice"],
                          val_metrics["precision"], val_metrics["recall"], val_metrics["f1"], elapsed])

    if val_metrics["iou"] > best_iou:
        best_iou = val_metrics["iou"]
        epochs_without_improvement = 0
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"  -> new best val IoU {best_iou:.4f}, checkpoint saved.")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"No improvement for {PATIENCE} epochs, stopping early at epoch {epoch}.")
            break

print(f"Training complete. Best val IoU: {best_iou:.4f}. Checkpoint at: {CHECKPOINT_PATH}")


## 9. Final evaluation + prediction visualization
Loads the best checkpoint (not just whatever's in memory at the end of training, in case early stopping wasn't the final epoch) and reports final metrics plus a few example predictions.

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()

metric_totals = {"iou": 0.0, "dice": 0.0, "precision": 0.0, "recall": 0.0, "f1": 0.0}
n_batches = 0
with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)
        logits = model(images)
        m = compute_metrics(logits, masks)
        for k in metric_totals:
            metric_totals[k] += m[k]
        n_batches += 1

final_metrics = {k: v / n_batches for k, v in metric_totals.items()}
print("Final validation metrics (best checkpoint):")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.4f}")


In [ ]:
# Visualize a handful of predictions vs ground truth
images, masks = next(iter(val_loader))
images_gpu = images.to(device)
with torch.no_grad():
    logits = model(images_gpu)
    preds = (torch.sigmoid(logits) > 0.5).float().cpu()

n = min(4, images.shape[0])
fig, axes = plt.subplots(3, n, figsize=(4 * n, 12))
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
for i in range(n):
    denorm = images[i] * std + mean
    axes[0, i].imshow(denorm.permute(1, 2, 0).clamp(0, 1).numpy())
    axes[0, i].set_title("input")
    axes[0, i].axis("off")
    axes[1, i].imshow(masks[i].squeeze(0).numpy(), cmap="gray")
    axes[1, i].set_title("ground truth")
    axes[1, i].axis("off")
    axes[2, i].imshow(preds[i].squeeze(0).numpy(), cmap="gray")
    axes[2, i].set_title("prediction")
    axes[2, i].axis("off")
plt.tight_layout()
plt.savefig("predictions_sample.png", dpi=120)
plt.show()


## 10. Download the trained weights

This is the step you don't want to skip before closing the tab. It packages the checkpoint, training log, and sample images into one zip and triggers a download.

**On Colab**, this pops a browser download automatically. **On local Jupyter**, the zip is just written to your working directory — no special download step needed, it's already on your machine.

In [ ]:
import shutil

bundle_name = "oil_spill_training_bundle"
shutil.make_archive(bundle_name, "zip", ".", "")  # fallback: zips whole cwd if selective copy below is skipped

# Prefer a clean, selective bundle instead of zipping everything in cwd:
import os
os.makedirs(bundle_name, exist_ok=True)
for fname in [CHECKPOINT_PATH, LOG_PATH, "sample_check.png", "predictions_sample.png"]:
    if os.path.exists(fname):
        shutil.copy(fname, bundle_name)

shutil.make_archive(bundle_name, "zip", bundle_name)
print(f"Bundle created: {bundle_name}.zip")

try:
    from google.colab import files
    files.download(f"{bundle_name}.zip")
    print("Download triggered via Colab.")
except ImportError:
    print(f"Not running on Colab — {bundle_name}.zip is already saved in your working directory:")
    print(os.path.abspath(f"{bundle_name}.zip"))


## Next steps

- Use `best_model.pth` (inside the downloaded zip) as the `--checkpoint` argument for your inference pipeline (`infer_pipeline.py`) to run predictions on real Mumbai Sentinel-1 tiles.
- Remember: the validation IoU/Dice reported here is on the **SOS dataset's own validation split**, not on Mumbai. Mumbai is a geographic domain shift — treat any Mumbai output as a qualitative demo, not a quantitatively validated accuracy claim, unless you have real Mumbai ground-truth masks to test against.
- If val IoU plateaus low, the first things worth checking (in order): (1) is `oil_fraction` from Step 5 extremely small — loss/metric imbalance handling may need tuning; (2) look at `predictions_sample.png` for a qualitative failure pattern (missed small spills? false positives on look-alike dark patches like calm water or land?); (3) only then consider a bigger encoder — not before the baseline is understood.
